# Downloader

This small script will help us download the images from the Azure Data Lake

## Install Dependencies

In [10]:
#r "nuget: Azure.Identity, 1.17.1"
#r "nuget: Azure.Storage.Blobs, 12.26.0"
#r "nuget: CsvHelper, 33.1.0"

Error: Azure.Storage.Blobs version 12.26.0 cannot be added because version 12.27.0-beta.1 was added previously.

## Add Usings

In [2]:
using System;
using System.Globalization;
using System.IO;
using Azure.Identity;
using Azure.Storage.Blobs;
using CsvHelper;
using CsvHelper.Configuration;

## Generate the CSV

To do it, we will execute the following SQL:
```sql
SELECT
    [Id],
    [Matricula],
    [LinkImagenVehiculo1],
    [LinkImagenVehiculo2]
FROM
    [HistoricoTransitoVehiculos]
```

# Parse the CSV

In [3]:
public record HistoricoTransitoVehiculos
{
    public long Id { get; init; }
    public string Matricula { get; set; }
    public string LinkImagenVehiculo1 { get; set; }
    public string LinkImagenVehiculo2 { get; set; }
}

var fileInfo = new FileInfo("historico-transito-vehiculos.csv");
var fileStream = fileInfo.Open(FileMode.Open, FileAccess.Read, FileShare.Read);
var streamReader = new StreamReader(fileStream);
var csvReader = new CsvReader(streamReader, new CsvConfiguration(CultureInfo.InvariantCulture) { HasHeaderRecord = true, Delimiter = ","});
var records = csvReader.GetRecords<HistoricoTransitoVehiculos>();

## Connect to Azure

In [ ]:
var connectionString = "";
var accountUri = new Uri("https://stgghlmtransauto01.blob.core.windows.net/");
var blobServiceClient = new BlobServiceClient(connectionString);
var blobContainerClient = blobServiceClient.GetBlobContainerClient("hmeta");

## Download Image

In [9]:
var directoryInfo = new DirectoryInfo("./images/");
directoryInfo.Create();

foreach (var historicoTransitoVehiculo in records.Take(10)) {
    var idDirectory = new DirectoryInfo($"{directoryInfo.FullName}{historicoTransitoVehiculo.Id}");
    idDirectory.Create();

    try {
        var fileInfo = new FileInfo($"{idDirectory.FullName}/{historicoTransitoVehiculo.Matricula}-link1.jpeg");
        if (!fileInfo.Exists) {
            var link = new Uri(historicoTransitoVehiculo.LinkImagenVehiculo1);
            var blobClient = blobContainerClient.GetBlobClient(link.AbsolutePath);
            blobClient.DownloadTo(fileInfo.FullName);
        }
    } catch (Exception exception) {
        Console.Out.WriteLine(exception.Message);
    }

    try {
        var fileInfo = new FileInfo($"{idDirectory.FullName}/{historicoTransitoVehiculo.Matricula}-link2.jpeg");
        if (!fileInfo.Exists) {
            var link = new Uri(historicoTransitoVehiculo.LinkImagenVehiculo2);
            var blobClient = blobContainerClient.GetBlobClient(link.AbsolutePath);
            blobClient.DownloadTo(fileInfo.FullName);
        }
    } catch (Exception exception) {
        Console.Out.WriteLine(exception.Message);
    }
}

This request is not authorized to perform this operation.
RequestId:1f140474-d01e-008e-6a03-74931f000000
Time:2025-12-23T11:58:06.9172021Z
Status: 403 (This request is not authorized to perform this operation.)
ErrorCode: AuthorizationFailure

Content:
﻿<?xml version="1.0" encoding="utf-8"?><Error><Code>AuthorizationFailure</Code><Message>This request is not authorized to perform this operation.
RequestId:1f140474-d01e-008e-6a03-74931f000000
Time:2025-12-23T11:58:06.9172021Z</Message></Error>

Headers:
Server: Microsoft-HTTPAPI/2.0
x-ms-request-id: 1f140474-d01e-008e-6a03-74931f000000
x-ms-client-request-id: 1bab7177-3374-4ad1-91f5-1f361acbe0c8
x-ms-error-code: AuthorizationFailure
Date: Tue, 23 Dec 2025 11:58:06 GMT
Content-Length: 246
Content-Type: application/xml

This request is not authorized to perform this operation.
RequestId:1f14047f-d01e-008e-7503-74931f000000
Time:2025-12-23T11:58:06.9611481Z
Status: 403 (This request is not authorized to perform this operation.)
ErrorCode: 